# P-gp inhibitor prediction - Phase 1: graph models against the Phase 0 baseline

Phase 0 (`results/phase0/`) benchmarked Morgan + RDKit descriptor models on 522
compounds and reached ROC-AUC **0.8065 [0.7581, 0.8481]**. The graph models it was
meant to be compared against were written and trained in July 2025, but their
outputs were cleared before the notebooks were archived and the Phase 1 result
directories were left empty, so no graph numbers survived.

This notebook recomputes both halves in one run, under a single protocol copied
from the Phase 0 Stage 2 notebook:

| | |
|---|---|
| dataset | 522 compounds, read from the repository over HTTPS |
| outer CV | `StratifiedKFold(5, shuffle=True, random_state=42)` |
| inner CV | `StratifiedKFold(4, shuffle=True, random_state=42)` |
| metric | ROC-AUC |
| interval | bootstrap over the 5 fold values, 1,000 resamples |

The outer split is a function of `(n_samples, y, seed)` alone, so the tabular and
graph models are scored on **identical test folds** - the comparison is paired.

### Protocol check

Cell 5 re-runs the Phase 0 Random Forest before anything else. If it does not
land back on 0.8065 the protocol has drifted, and the graph numbers below it
should not be read.

### Layout

Each analysis cell writes its result under `checkpoints/`. Re-running a completed
cell reloads it, so an interrupted session continues where it stopped.

| Cell | Contents | Checkpoint |
|---|---|---|
| 1 | Environment, configuration, checkpoint I/O | - |
| 2 | Data | `data` |
| 3 | Morgan / RDKit features, molecular graphs | `features` |
| 4 | Protocol: folds, metrics, bootstrap | - |
| 5 | Random Forest baseline (protocol check) | `tabular` |
| 6 | GCN, GAT, GINE | `gnn` |
| 7 | Summary report | - |

Expect roughly 1-3 hours with `FULL_GRID = True`; the tabular grid is the slow
part, not the graph models.

In [1]:
# Cell 1 - Environment, configuration, checkpoint I/O
!pip -q install rdkit torch-geometric

import os, json, pickle, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ---- what to run -------------------------------------------------
# The tabular grid is the one Phase 0 used: 216 combinations x 4 inner folds x
# 5 outer folds, and it is the slow part. FULL_GRID = False swaps in a reduced
# grid that finishes in minutes but will NOT reproduce 0.8065.
RUN_BASELINE     = True     # Random Forest, reproduces the Phase 0 baseline
RUN_GNN          = True     # GCN, GAT, GINE
FULL_GRID        = True

# ---- storage -----------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/pgp_phase1_results'
except Exception:
    BASE = './pgp_phase1_results'
CKPT = os.path.join(BASE, 'checkpoints')
os.makedirs(CKPT, exist_ok=True)

def _p(name):
    return os.path.join(CKPT, name + '.pkl')

def save_ckpt(name, obj):
    tmp = _p(name) + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    os.replace(tmp, _p(name))            # atomic: never a half-written file

def load_ckpt(name, default=None):
    if os.path.exists(_p(name)):
        with open(_p(name), 'rb') as f:
            return pickle.load(f)
    return default

def have(name):
    return os.path.exists(_p(name))

def ckpt_status():
    print(f"{'checkpoint':<12}{'saved':<8}size")
    for n in ['data', 'features', 'tabular', 'gnn']:
        p = _p(n)
        size = f'{os.path.getsize(p)/1e6:.1f} MB' if os.path.exists(p) else '-'
        print(f"{n:<12}{'YES' if os.path.exists(p) else 'no':<8}{size}")

# ---- protocol constants, copied from the Phase 0 Stage 2 notebook ----
CONFIG = dict(CV_FOLDS=5, N_BOOTSTRAP=1000, MORGAN_RADIUS=2, MORGAN_BITS=2048)
RDKIT_DESCRIPTORS = [
    'MolWt', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors',
    'NumRotatableBonds', 'NumAromaticRings', 'NumSaturatedRings',
    'NumHeteroatoms', 'RingCount', 'FractionCsp3', 'BertzCT',
]
CSV_URL = ('https://raw.githubusercontent.com/Jay99Sohn/'
           'pgp-literature-mining-inhibitor-prediction/main/data/processed/'
           'pgp_dataset_for_inhibitor_model_with_smiles_522.csv')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('\nconfig :', json.dumps(CONFIG))
print('storage:', BASE)
print('device :', DEVICE)
ckpt_status()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.3 MB/s eta 0:00:00
Mounted at /content/drive

config : {"CV_FOLDS": 5, "N_BOOTSTRAP": 1000, "MORGAN_RADIUS": 2, "MORGAN_BITS": 2048}
storage: /content/drive/MyDrive/pgp_phase1_results
device : cuda
checkpoint  saved   size
data        no      -
features    no      -
tabular     no      -
gnn         no      -


In [2]:
# Cell 2 - Data
# Checkpoint: data
if have('data'):
    D = load_ckpt('data')
    print("[SKIP] loaded 'data' from checkpoint")
else:
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog('rdApp.*')

    df = pd.read_csv(CSV_URL)
    df = df[df['smiles'].notna() & (df['smiles'].astype(str).str.strip() != '')]
    df = df.reset_index(drop=True)
    df['label'] = df['label'].astype(int)

    # 34 cells carry a second structure on a following line. RDKit stops at the
    # newline and returns the first molecule either way; normalising here keeps
    # the column well-formed on checkouts that rewrite line endings.
    df['smiles'] = df['smiles'].astype(str).str.strip().str.split('\n').str[0].str.strip()

    n_ok = sum(Chem.MolFromSmiles(s) is not None for s in df['smiles'])
    print(f'[QC] compounds {len(df)}  '
          f'(inhibitors {int(df["label"].sum())} / non {int((df["label"] == 0).sum())})')
    print(f'[QC] parsed    {n_ok}/{len(df)}')
    assert n_ok == len(df), 'some SMILES did not parse'

    D = dict(df=df)
    save_ckpt('data', D)

df = D['df']
y = df['label'].values
print(f"\nsamples {len(y)}  |  inhibitors {int(y.sum())}  |  non {int((y == 0).sum())}")

[QC] compounds 522  (inhibitors 177 / non 345)
[QC] parsed    522/522

samples 522  |  inhibitors 177  |  non 345


In [3]:
# Cell 3 - Morgan / RDKit features and molecular graphs
# Checkpoint: features
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.preprocessing import StandardScaler
RDLogger.DisableLog('rdApp.*')

GETTERS = {
    'MolWt': Descriptors.MolWt, 'LogP': Descriptors.MolLogP,
    'TPSA': Descriptors.TPSA, 'NumHDonors': Descriptors.NumHDonors,
    'NumHAcceptors': Descriptors.NumHAcceptors,
    'NumRotatableBonds': Descriptors.NumRotatableBonds,
    'NumAromaticRings': Descriptors.NumAromaticRings,
    'NumSaturatedRings': Descriptors.NumSaturatedRings,
    'NumHeteroatoms': Descriptors.NumHeteroatoms,
    'RingCount': Descriptors.RingCount,
    'FractionCsp3': Descriptors.FractionCSP3, 'BertzCT': Descriptors.BertzCT,
}

def tabular():
    '''Morgan fingerprint + standardized RDKit descriptors, concatenated.'''
    morgan, desc = [], []
    for smi in df['smiles']:
        mol = Chem.MolFromSmiles(str(smi))
        fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(
            mol, radius=CONFIG['MORGAN_RADIUS'], nBits=CONFIG['MORGAN_BITS'])
        morgan.append(np.array(fp, dtype=int))
        desc.append([float(GETTERS[n](mol)) for n in RDKIT_DESCRIPTORS])
    X_m = np.asarray(morgan)
    X_d = np.nan_to_num(StandardScaler().fit_transform(np.asarray(desc, dtype=float)),
                        nan=0.0, posinf=0.0, neginf=0.0)
    return np.hstack([X_m, X_d])

ELEMENTS = [6, 7, 8, 9, 15, 16, 17, 35, 53]          # C N O F P S Cl Br I
HYBRIDS = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
           Chem.rdchem.HybridizationType.SP3]
BONDS = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
         Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

def atom_feat(a):
    z = [float(a.GetAtomicNum() == e) for e in ELEMENTS]
    z.append(float(a.GetAtomicNum() not in ELEMENTS))
    h = [float(a.GetHybridization() == t) for t in HYBRIDS]
    return z + h + [a.GetDegree() / 4.0, float(a.GetFormalCharge()),
                    a.GetTotalNumHs() / 4.0, float(a.GetIsAromatic()),
                    float(a.IsInRing())]

def bond_feat(b):
    return ([float(b.GetBondType() == t) for t in BONDS]
            + [float(b.GetIsConjugated()), float(b.IsInRing())])

def build_graphs():
    '''SMILES -> PyG Data. Bond features are included because GINEConv consumes
    edge attributes; that is the whole difference between GINE and plain GIN.'''
    from torch_geometric.data import Data
    graphs = []
    for smi, lab in zip(df['smiles'], df['label']):
        mol = Chem.MolFromSmiles(smi)
        x = torch.tensor([atom_feat(a) for a in mol.GetAtoms()], dtype=torch.float)
        src, dst, ea = [], [], []
        for b in mol.GetBonds():
            i, j, f = b.GetBeginAtomIdx(), b.GetEndAtomIdx(), bond_feat(b)
            src += [i, j]
            dst += [j, i]                                  # undirected
            ea += [f, f]
        if src:
            ei = torch.tensor([src, dst], dtype=torch.long)
            eattr = torch.tensor(ea, dtype=torch.float)
        else:                                              # single-atom molecule
            ei = torch.zeros((2, 0), dtype=torch.long)
            eattr = torch.zeros((0, len(BONDS) + 2), dtype=torch.float)
        graphs.append(Data(x=x, edge_index=ei, edge_attr=eattr,
                           y=torch.tensor([float(lab)])))
    return graphs

if have('features'):
    F = load_ckpt('features')
    print("[SKIP] loaded 'features' from checkpoint")
else:
    F = dict(X=tabular())
    save_ckpt('features', F)

GRAPHS = build_graphs()          # cheap, so rebuilt each session rather than pickled
print(f"\ntabular  {F['X'].shape}")
print(f"graphs   {len(GRAPHS)}  |  atom features {GRAPHS[0].x.shape[1]}"
      f"  |  bond features {GRAPHS[0].edge_attr.shape[1]}")


tabular  (522, 2060)
graphs   522  |  atom features 18  |  bond features 6


In [4]:
# Cell 4 - Protocol: folds, metrics, bootstrap
# Copied from the Phase 0 Stage 2 notebook so the numbers stay comparable.
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, matthews_corrcoef, precision_score,
                             recall_score, f1_score, cohen_kappa_score)

OUTER = list(StratifiedKFold(n_splits=CONFIG['CV_FOLDS'], shuffle=True,
                             random_state=SEED).split(np.zeros(len(y)), y))

def inner_cv():
    return StratifiedKFold(n_splits=CONFIG['CV_FOLDS'] - 1, shuffle=True,
                           random_state=SEED)

def fold_metrics(yt, yp, proba):
    return {'roc_auc': roc_auc_score(yt, proba),
            'mcc': matthews_corrcoef(yt, yp),
            'precision': precision_score(yt, yp, zero_division=0),
            'recall': recall_score(yt, yp, zero_division=0),
            'f1': f1_score(yt, yp, zero_division=0),
            'kappa': cohen_kappa_score(yt, yp)}

def summarize(folds):
    '''Phase 0 resampled the five fold-level scores rather than the individual
    predictions. Reproduced as-is: changing it would make the interval
    incomparable with the published baseline.'''
    rng = np.random.RandomState(SEED)
    out = {}
    for k in folds[0]:
        if k in ('best_params', 'fold'):
            continue
        v = np.array([f[k] for f in folds], dtype=float)
        boot = [v[rng.randint(0, len(v), len(v))].mean()
                for _ in range(CONFIG['N_BOOTSTRAP'])]
        out[k] = dict(mean=float(v.mean()), std=float(v.std()),
                      ci=(float(np.percentile(boot, 2.5)),
                          float(np.percentile(boot, 97.5))))
    return out

print('outer folds', [(len(a), len(b)) for a, b in OUTER])
print('positive rate per test fold', ['%.3f' % y[b].mean() for a, b in OUTER])
print('the same folds are used for the tabular and the graph models')

outer folds [(417, 105), (417, 105), (418, 104), (418, 104), (418, 104)]
positive rate per test fold ['0.343', '0.343', '0.337', '0.337', '0.337']
the same folds are used for the tabular and the graph models


In [5]:
# Cell 5 - Random Forest baseline (Phase 0 protocol check)
# Checkpoint: tabular
RF_GRID_FULL = {'n_estimators': [100, 200, 500],
                'max_depth': [None, 10, 20, 30],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf': [1, 2, 4],
                'class_weight': ['balanced', None]}
RF_GRID_QUICK = {'n_estimators': [200], 'max_depth': [None, 20],
                 'class_weight': ['balanced', None]}
GRID = RF_GRID_FULL if FULL_GRID else RF_GRID_QUICK

def run_rf(X):
    folds = []
    for k, (tr, te) in enumerate(OUTER):
        gs = GridSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=-1),
                          GRID, cv=inner_cv(), scoring='roc_auc', n_jobs=-1)
        gs.fit(X[tr], y[tr])
        est = gs.best_estimator_
        m = fold_metrics(y[te], est.predict(X[te]), est.predict_proba(X[te])[:, 1])
        m['best_params'] = gs.best_params_
        m['fold'] = k
        folds.append(m)
        print(f'      fold {k + 1}/{CONFIG["CV_FOLDS"]}  AUC {m["roc_auc"]:.4f}')
    return {'summary': summarize(folds), 'fold_results': folds}

tab = load_ckpt('tabular', {})
t0 = time.time()

if RUN_BASELINE and 'RandomForest' not in tab:
    print('RandomForest  (Phase 0 baseline, protocol check)')
    tab['RandomForest'] = run_rf(F['X'])
    save_ckpt('tabular', tab)

print(f'\n({time.time() - t0:.0f}s)')
for name, r in tab.items():
    s = r['summary']['roc_auc']
    print(f"  {name:<32} AUC {s['mean']:.4f}  [{s['ci'][0]:.4f}, {s['ci'][1]:.4f}]")

if FULL_GRID and 'RandomForest' in tab:

    got = tab['RandomForest']['summary']['roc_auc']['mean']
    print(f'\nprotocol check: {got:.4f} against 0.8065 published')
    print('  reproduces the published baseline' if abs(got - 0.8065) <= 0.01 else
          '  *** does not reproduce - resolve this before reporting the graph numbers ***')

RandomForest  (Phase 0 baseline, protocol check)
      fold 1/5  AUC 0.8293
      fold 2/5  AUC 0.7150
      fold 3/5  AUC 0.7799
      fold 4/5  AUC 0.8501
      fold 5/5  AUC 0.8476

(2537s)
  RandomForest                     AUC 0.8044  [0.7550, 0.8450]

protocol check: 0.8044 against 0.8065 published
  reproduces the published baseline


In [6]:
# Cell 6 - GCN, GAT, GINE
# Checkpoint: gnn
import torch.nn as nn
import torch.nn.functional as Fn
from torch_geometric.nn import GCNConv, GATConv, GINEConv, global_mean_pool
from torch_geometric.loader import DataLoader

GNN_GRID = [{'hidden': 64, 'lr': 1e-3}, {'hidden': 64, 'lr': 5e-4},
            {'hidden': 128, 'lr': 1e-3}, {'hidden': 128, 'lr': 5e-4}]
MAX_EPOCHS, PATIENCE, BATCH = 200, 20, 32

class Net(nn.Module):
    '''Same depth and width for all three; only the convolution changes.'''

    def __init__(self, kind, in_dim, edge_dim, hidden):
        super().__init__()
        self.kind = kind
        if kind == 'GCN':                       # GCNConv takes no edge features
            self.c1 = GCNConv(in_dim, hidden)
            self.c2 = GCNConv(hidden, hidden)
            self.c3 = GCNConv(hidden, hidden)
        elif kind == 'GAT':
            h = 4
            self.c1 = GATConv(in_dim, hidden // h, heads=h, edge_dim=edge_dim)
            self.c2 = GATConv(hidden, hidden // h, heads=h, edge_dim=edge_dim)
            self.c3 = GATConv(hidden, hidden, heads=1, edge_dim=edge_dim)
        elif kind == 'GINE':
            def mlp(i, o):
                return nn.Sequential(nn.Linear(i, o), nn.ReLU(), nn.Linear(o, o))
            self.c1 = GINEConv(mlp(in_dim, hidden), edge_dim=edge_dim)
            self.c2 = GINEConv(mlp(hidden, hidden), edge_dim=edge_dim)
            self.c3 = GINEConv(mlp(hidden, hidden), edge_dim=edge_dim)
        else:
            raise ValueError(kind)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(hidden, 1))

    def forward(self, d):
        x, ei, ea = d.x, d.edge_index, d.edge_attr
        for c in (self.c1, self.c2, self.c3):
            x = Fn.relu(c(x, ei) if self.kind == 'GCN' else c(x, ei, ea))
        return self.head(global_mean_pool(x, d.batch)).squeeze(-1)

def predict(model, idx):
    model.eval()
    ps, ts = [], []
    with torch.no_grad():
        for b in DataLoader([GRAPHS[i] for i in idx], batch_size=256):
            b = b.to(DEVICE)
            ps.append(torch.sigmoid(model(b)).cpu().numpy())
            ts.append(b.y.view(-1).cpu().numpy())
    return np.concatenate(ts), np.concatenate(ps)

def train(kind, tr, va, cfg, epochs):
    torch.manual_seed(SEED)
    model = Net(kind, GRAPHS[0].x.shape[1], GRAPHS[0].edge_attr.shape[1],
                cfg['hidden']).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    ytr = y[tr]
    pw = torch.tensor([(ytr == 0).sum() / max((ytr == 1).sum(), 1)],
                      dtype=torch.float, device=DEVICE)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    tl = DataLoader([GRAPHS[i] for i in tr], batch_size=BATCH, shuffle=True)

    best, best_ep, stale = -1.0, epochs, 0
    for ep in range(1, epochs + 1):
        model.train()
        for b in tl:
            b = b.to(DEVICE)
            opt.zero_grad()
            lossf(model(b), b.y.view(-1)).backward()
            opt.step()
        if va is None:
            continue
        t, p = predict(model, va)
        auc = roc_auc_score(t, p) if len(np.unique(t)) > 1 else 0.5
        if auc > best:
            best, best_ep, stale = auc, ep, 0
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    return model, best, best_ep

def run_gnn(kind):
    folds = []
    for k, (tr, te) in enumerate(OUTER):
        best_cfg, best_score, best_ep = None, -1.0, MAX_EPOCHS
        for cfg in GNN_GRID:                    # inner CV, training data only
            sc, eps = [], []
            for itr, iva in inner_cv().split(np.zeros(len(tr)), y[tr]):
                _, auc, ep = train(kind, tr[itr], tr[iva], cfg, MAX_EPOCHS)
                sc.append(auc)
                eps.append(ep)
            if float(np.mean(sc)) > best_score:
                best_cfg, best_score = cfg, float(np.mean(sc))
                best_ep = max(1, int(round(float(np.mean(eps)))))
        # Refit on the whole outer-training fold for the epoch budget the inner
        # folds chose, so the test fold never influences stopping.
        model, _, _ = train(kind, tr, None, best_cfg, best_ep)
        t, p = predict(model, te)
        m = fold_metrics(t, (p >= 0.5).astype(int), p)
        m['best_params'] = {**best_cfg, 'epochs': best_ep}
        m['fold'] = k
        folds.append(m)
        print(f'      fold {k + 1}/{CONFIG["CV_FOLDS"]}  AUC {m["roc_auc"]:.4f}  '
              f'(hidden={best_cfg["hidden"]}, lr={best_cfg["lr"]}, epochs={best_ep})')
    return {'summary': summarize(folds), 'fold_results': folds}

gnn = load_ckpt('gnn', {})
if RUN_GNN:
    for kind in ['GCN', 'GAT', 'GINE']:
        if kind in gnn:
            print(f'[SKIP] {kind} already done')
            continue
        print(kind)
        t0 = time.time()
        gnn[kind] = run_gnn(kind)
        save_ckpt('gnn', gnn)
        print(f'   ({time.time() - t0:.0f}s)')

for name, r in gnn.items():
    s = r['summary']['roc_auc']
    print(f"  {name:<32} AUC {s['mean']:.4f}  [{s['ci'][0]:.4f}, {s['ci'][1]:.4f}]")

GCN
      fold 1/5  AUC 0.6848  (hidden=64, lr=0.001, epochs=91)
      fold 2/5  AUC 0.6977  (hidden=64, lr=0.001, epochs=125)
      fold 3/5  AUC 0.6886  (hidden=128, lr=0.001, epochs=88)
      fold 4/5  AUC 0.7420  (hidden=128, lr=0.001, epochs=103)
      fold 5/5  AUC 0.7499  (hidden=64, lr=0.001, epochs=61)
   (688s)
GAT
      fold 1/5  AUC 0.7335  (hidden=128, lr=0.001, epochs=54)
      fold 2/5  AUC 0.7138  (hidden=128, lr=0.001, epochs=61)
      fold 3/5  AUC 0.7801  (hidden=64, lr=0.001, epochs=77)
      fold 4/5  AUC 0.8199  (hidden=64, lr=0.001, epochs=92)
      fold 5/5  AUC 0.7346  (hidden=128, lr=0.001, epochs=82)
   (1037s)
GINE
      fold 1/5  AUC 0.5874  (hidden=64, lr=0.001, epochs=54)
      fold 2/5  AUC 0.6260  (hidden=128, lr=0.001, epochs=47)
      fold 3/5  AUC 0.5706  (hidden=128, lr=0.001, epochs=49)
      fold 4/5  AUC 0.6816  (hidden=128, lr=0.001, epochs=54)
      fold 5/5  AUC 0.7010  (hidden=128, lr=0.0005, epochs=92)
   (528s)
  GCN                        

In [7]:
# Cell 7 - Summary report
from datetime import datetime

tab = load_ckpt('tabular', {})
gnn = load_ckpt('gnn', {})
allr = {**tab, **gnn}
if not allr:
    raise RuntimeError('nothing to report - run Cell 5 and/or Cell 6 first')

rows = []
for name, r in allr.items():
    s = r['summary']
    rows.append({'model': name,
                 'roc_auc': round(s['roc_auc']['mean'], 4),
                 'sd': round(s['roc_auc']['std'], 4),
                 'ci_lower': round(s['roc_auc']['ci'][0], 4),
                 'ci_upper': round(s['roc_auc']['ci'][1], 4),
                 'mcc': round(s['mcc']['mean'], 4),
                 'f1': round(s['f1']['mean'], 4),
                 'precision': round(s['precision']['mean'], 4),
                 'recall': round(s['recall']['mean'], 4)})
table = pd.DataFrame(rows).sort_values('roc_auc', ascending=False)
table.to_csv(os.path.join(BASE, 'phase1_benchmark_results.csv'), index=False)

folds = pd.DataFrame(
    [{'model': name,
      **{f'fold_{f["fold"] + 1}': round(f['roc_auc'], 4) for f in r['fold_results']}}
     for name, r in allr.items()])
folds = folds.set_index('model').loc[table['model']].reset_index()
folds.to_csv(os.path.join(BASE, 'phase1_fold_auc.csv'), index=False)

L = []
A = L.append
A('=' * 78)
A('Phase 1: graph neural networks against the Phase 0 tabular baseline')
A('=' * 78)
A('')
A('PROTOCOL  (copied from the Phase 0 Stage 2 notebook)')
A(f'  dataset            {len(y)} compounds '
  f'({int(y.sum())} inhibitors / {int((y == 0).sum())} non-inhibitors)')
A(f'  outer CV           StratifiedKFold({CONFIG["CV_FOLDS"]}, shuffle=True, '
  f'random_state={SEED})')
A(f'  inner CV           StratifiedKFold({CONFIG["CV_FOLDS"] - 1}, ...) '
  f'grid search on ROC-AUC')
A(f'  interval           bootstrap over the {CONFIG["CV_FOLDS"]} fold values, '
  f'{CONFIG["N_BOOTSTRAP"]} resamples')
A('  class imbalance    class weighting (tabular) / pos_weight (graph)')
A('  tabular features   Morgan r=2, 2048 bits + 12 standardized RDKit descriptors')
A('  graph features     atom: element, hybridization, degree, charge, H count,')
A('                     aromaticity, ring')
A('                     bond: type, conjugation, ring - used by GAT and GINE;')
A('                     GCNConv takes no edge features')
A(f'  hyperparameters    graph models searched over {len(GNN_GRID)} configurations')
A('')
A('RESULT')
for line in table.to_string(index=False).split('\n'):
    A('  ' + line)
A('')
A('PER-FOLD ROC-AUC')
for line in folds.to_string(index=False).split('\n'):
    A('  ' + line)
A('')
if 'RandomForest' in allr:
    got = allr['RandomForest']['summary']['roc_auc']
    diff = abs(got['mean'] - 0.8065)
    A('PROTOCOL CHECK')
    A('  Phase 0 published   0.8065  [0.7581, 0.8481]')
    A(f'  re-run here         {got["mean"]:.4f}  '
      f'[{got["ci"][0]:.4f}, {got["ci"][1]:.4f}]')
    if diff <= 0.01:
        A(f'  The baseline reproduces to within {diff:.3f}, so the folds, features and')
        A('  interval are the ones the published figure came from, and the graph models')
        A('  below are measured on the same scale.')
    else:
        A(f'  *** The baseline differs from the published figure by {diff:.3f}. The')
        A('  protocol has drifted; the graph numbers below should not be reported')
        A('  until that is resolved. ***')
    A('')

A('NOTES')
A('  Tabular and graph models are scored on identical outer folds, so the')
A('  comparison is paired.')

graph_models = [m for m in allr if m != 'RandomForest']
if 'RandomForest' in allr and graph_models:
    from scipy.stats import ttest_rel
    rf_folds = np.array([f['roc_auc'] for f in allr['RandomForest']['fold_results']])
    rf_mean = allr['RandomForest']['summary']['roc_auc']['mean']
    beat = [m for m in graph_models
            if allr[m]['summary']['roc_auc']['mean'] > rf_mean]
    A('  ' + ('No graph model reached the baseline.' if not beat else
               'Above the baseline: ' + ', '.join(beat) + '.'))
    A('  Paired against the baseline on the same folds:')
    for m in graph_models:
        gm = np.array([f['roc_auc'] for f in allr[m]['fold_results']])
        d = rf_folds - gm
        _, p = ttest_rel(rf_folds, gm)
        A(f'    {m:<6} mean difference {d.mean():+.4f}, baseline ahead on '
          f'{int((d > 0).sum())}/{len(d)} folds, p = {p:.3f}')
    A('  Five folds resolve only large differences; a p above ~0.05 means the models')
    A('  are not separated by this comparison, not that they perform equally.')
    worst = min(graph_models, key=lambda m: allr[m]['summary']['roc_auc']['mean'])
    A(f'  Weakest graph model: {worst}.')
    if worst == 'GINE':
        A('  GINE carries the richest representation - bond features and an MLP')
        A('  aggregator - yet ranks last, so on 522 molecules the extra capacity')
        A('  costs more than the representation returns.')

A('  Labels are literature-derived rather than assay-standardized; these models')
A('  rank hypotheses rather than predict measured activity.')
A('  The graph models were written and trained in July 2025 but their outputs were')
A('  not retained, so the numbers above are a re-run under the Phase 0 protocol,')
A('  not a recovery of the original ones.')
A('')
A(f'  generated  {datetime.now().strftime("%Y-%m-%d")}')
A('')

report = '\n'.join(L)
with open(os.path.join(BASE, 'phase1_report.txt'), 'w') as f:
    f.write(report)
print(report)
print('wrote', BASE)

Phase 1: graph neural networks against the Phase 0 tabular baseline

PROTOCOL  (copied from the Phase 0 Stage 2 notebook)
  dataset            522 compounds (177 inhibitors / 345 non-inhibitors)
  outer CV           StratifiedKFold(5, shuffle=True, random_state=42)
  inner CV           StratifiedKFold(4, ...) grid search on ROC-AUC
  interval           bootstrap over the 5 fold values, 1000 resamples
  class imbalance    class weighting (tabular) / pos_weight (graph)
  tabular features   Morgan r=2, 2048 bits + 12 standardized RDKit descriptors
  graph features     atom: element, hybridization, degree, charge, H count,
                     aromaticity, ring
                     bond: type, conjugation, ring - used by GAT and GINE;
                     GCNConv takes no edge features
  hyperparameters    graph models searched over 4 configurations

RESULT
         model  roc_auc     sd  ci_lower  ci_upper    mcc     f1  precision  recall
  RandomForest   0.8044 0.0513    0.7550    0.8450